# Importing libraries

In [43]:
import pandas as pd
from sklearn import svm
import seaborn as sns
import matplotlib.pyplot as plt

In [44]:
df=pd.read_csv("data/data.csv")
df

,Gene_Symbol,rsID,Chromosome,Position,Ref_Allele,Alt_Allele,Variant_Type,Variant_Type_Label,Non_Coding_Region,Non_Coding_Region_Label,...,TF_FOXM1_disrupted,TF_FOXM1_created,TF_FOSL2_disrupted,TF_FOSL2_created,TF_Count_Disrupted,TF_Count_Created,Max_Disruption_Score,Most_Disrupted_TF,Most_Created_TF,Motif_Score_Reliable
0,BRCA1,rs8176320,Chr17,43044346,C,G,0,single nucleotide variant,0,3_prime_UTR,...,0,0,0,0,0,0,0.0,NaN,NaN,1
1,BRCA1,rs12516,Chr17,43044391,G,A,0,single nucleotide variant,0,3_prime_UTR,...,0,0,0,0,0,0,0.0,NaN,NaN,1
2,BRCA1,rs548275991,Chr17,43044392,G,A,0,single nucleotide variant,0,3_prime_UTR,...,0,0,0,0,0,0,0.0,NaN,NaN,1
3,BRCA1,rs8176319,Chr17,43044897,G,A,0,single nucleotide variant,0,3_prime_UTR,...,0,0,0,0,0,0,0.0,NaN,NaN,1
4,BRCA1,rs8176318,Chr17,43045257,C,A,0,single nucleotide variant,0,3_prime_UTR,...,0,0,0,0,0,0,0.0,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2396,CHEK2,rs1033667,Chr22,28734312,C,T,0,single nucleotide variant,2,intronic,...,0,0,0,0,0,0,0.0,NaN,NaN,1
2397,CHEK2,rs3841692,Chr22,28734360,NaN,NaN,1,indel,2,intronic,...,0,0,0,0,0,0,0.0,NaN,NaN,0
2398,CHEK2,rs587780181,Chr22,28734462,NaN,NaN,1,indel,1,5_prime_UTR,...,0,0,0,0,0,0,0.0,NaN,NaN,0
2399,CHEK2,rs2236141,Chr22,28741882,C,T,0,single nucleotide variant,4,upstream,...,0,0,0,0,0,0,0.0,NaN,NaN,1


# Data Exploration, Preprocessing


Note:
Variant_Type: 0=SNV, 1=indel, 2=deletion, 3=insertion, 4=other

Non_Coding_Region: 0=3'UTR, 1=5'UTR, 2=intronic, 3=splice region, 4=upstream, 5=downstream, 6=intergenic, 7=nc transcript, 8=regulatory, 9=other

ClinVar_Significance: -1=missing, 0=benign, 1=likely-benign, 2=uncertain, 3=likely-pathogenic, 4=pathogenic

In [45]:
df["ClinVar_Significance_Label"].unique()
df["ClinVar_Significance_Label"].value_counts()

ClinVar_Significance_Label
likely-benign                                   1079
uncertain-significance                           627
benign                                           314
conflicting-interpretations-of-pathogenicity     148
pathogenic                                       106
likely-pathogenic                                 73
not-provided                                      30
benign-likely-benign                              13
pathogenic-likely-pathogenic                      11
Name: count, dtype: int64

In [46]:
df["ClinVar_Significance"].value_counts()

ClinVar_Significance
1    1079
2     805
0     327
4     106
3      84
Name: count, dtype: int64

* Drop unlabelled/uncertain data
* Remove columns with single-valued or null-valued values:

In [47]:
labels_out =  ["uncertain-significance", "not-provided", "conflicting-interpretations-of-pathogenicity"]
data = df.drop(df[df["ClinVar_Significance_Label"].isin(labels_out)].index)
data["ClinVar_Significance_Label"].value_counts()
for col in data.columns:
    if(len(data[col].value_counts())==1):
        data.drop(col, axis=1, inplace=True)


In [55]:
for col in data.columns:
    print(f"--------- Value counts for {col}")
    print(len(data[col].value_counts()))

--------- Value counts for Gene_Symbol
7
--------- Value counts for rsID
1596
--------- Value counts for Chromosome
6
--------- Value counts for Position
1514
--------- Value counts for Ref_Allele
39
--------- Value counts for Alt_Allele
39
--------- Value counts for Variant_Type
5
--------- Value counts for Variant_Type_Label
4
--------- Value counts for Non_Coding_Region
6
--------- Value counts for Non_Coding_Region_Label
6
--------- Value counts for H3K4me3
2
--------- Value counts for H3K27ac
2
--------- Value counts for Top_TF_Binding
5
--------- Value counts for ClinVar_Significance
4
--------- Value counts for ClinVar_Significance_Label
6
--------- Value counts for GTEx_eQTL
2
--------- Value counts for Known_Regulatory_Effect
2
--------- Value counts for Training_Split
3
--------- Value counts for H3K4me3_PeakCount
2
--------- Value counts for H3K27ac_CellLines
0
--------- Value counts for H3K4me3_PeakScore
4
--------- Value counts for H3K27ac_PeakScore
12
--------- Value coun

In [56]:
data["ClinVar_Significance_Label"].value_counts()

ClinVar_Significance_Label
likely-benign                   1079
benign                           314
pathogenic                       106
likely-pathogenic                 73
benign-likely-benign              13
pathogenic-likely-pathogenic      11
Name: count, dtype: int64